# Pré-processamento do Dataset de Soja

Nesta etapa vamos preparar os dados para o modelo de visão computacional.

O fluxo será:

1. Baixar os datasets via KaggleHub;
2. Inspecionar as pastas originais;
3. Organizar as imagens em uma estrutura única por classe;
4. Criar divisão treino, validação e teste;
5. Reduzir excesso da classe saudável no treino;
6. Aplicar augmentação offline nas classes doentes;
7. Conferir a distribuição final;
8. Preparação dos dados para treinamento.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from src.preprocessing.soja_preprocessing import (
    baixar_datasets,
    listar_pastas_dataset,
    contar_imagens_por_classe,
    organizar_dataset_soja,
    mostrar_distribuicao_dataset,
    criar_split_dataset,
    limitar_classe_saudavel_treino,
    aplicar_augmentacao_treino,
    mostrar_distribuicao_split,
    DATASET_ORGANIZADO,
    DATASET_SPLIT
)

from src.dataset.soja_dataset import (
    criar_datasets,
    criar_dataloaders,
    mostrar_info_datasets,
    obter_classes,
    obter_class_to_idx,
    obter_idx_to_class,
    DATASET_SPLIT
)

## 1. Download dos datasets

Vamos baixar dois datasets:

- Dataset de folhas de soja doentes;
- Dataset PlantVillage, usado para obter imagens de soja saudável.

In [0]:
pasta_soja_doente, pasta_plantvillage = baixar_datasets()

## 2. Inspeção inicial dos datasets

Antes de transformar os dados, é importante verificar a estrutura das pastas originais.

Isso ajuda a confirmar se os caminhos e nomes das classes estão corretos.

In [0]:
listar_pastas_dataset(pasta_soja_doente)

In [0]:
listar_pastas_dataset(pasta_plantvillage)

In [0]:
listar_pastas_dataset(pasta_plantvillage / "PlantVillage")

In [0]:
listar_pastas_dataset(pasta_plantvillage / "PlantVillage" / "train")

## 3. Organização do dataset

Agora vamos criar um dataset único de soja.

As imagens serão organizadas assim:

```
dataset_soja/
    Soja_Saudavel/
    Ferrugem/
    Mancha_Parda/
    Oidio/
    ...
```

Nesta etapa também fazemos a padronização dos nomes das classes.

In [0]:
organizar_dataset_soja(
    pasta_soja_doente=pasta_soja_doente,
    pasta_plantvillage=pasta_plantvillage,
    limpar_destino=True
)

In [0]:
mostrar_distribuicao_dataset(DATASET_ORGANIZADO)

## 4. Split treino, validação e teste

Agora vamos dividir o dataset em:

- `train`: usado para treinar o modelo;
- `val`: usado para acompanhar o desempenho durante o treinamento;
- `test`: usado somente para avaliação final.

A divisão configurada no arquivo é:

- 70% treino;
- 15% validação;
- 15% teste.

In [0]:
criar_split_dataset(
    pasta_origem=DATASET_ORGANIZADO,
    pasta_destino=DATASET_SPLIT,
    limpar_destino=True
)

In [0]:
mostrar_distribuicao_split(DATASET_SPLIT)

## 5. Redução da classe saudável no treino

A classe saudável pode ter muito mais imagens do que algumas classes doentes.

Para reduzir o desbalanceamento inicial, vamos limitar a quantidade de imagens saudáveis no treino.

In [0]:
limitar_classe_saudavel_treino(
    pasta_split=DATASET_SPLIT,
    max_imagens=100
)

## 6. Augmentação offline nas classes doentes

Agora vamos aumentar artificialmente as classes doentes no conjunto de treino.

A augmentação aplica:

- rotação aleatória;
- espelhamento horizontal;
- alteração de brilho.

A classe saudável será ignorada nessa etapa.

In [0]:
aplicar_augmentacao_treino(
    pasta_split=DATASET_SPLIT,
    alvo_por_classe=400
)

## 7. Distribuição final

Por fim, vamos conferir a quantidade final de imagens por classe em cada split.

In [0]:
mostrar_distribuicao_split(DATASET_SPLIT)

## 8. Preparação dos dados para treinamento

Nesta etapa vamos criar os `transforms`, os `ImageFolder` e os `DataLoaders`.

O objetivo é validar se a estrutura final do dataset está correta e se o PyTorch consegue carregar as imagens normalmente.



Agora vamos criar os datasets de treino, validação e teste usando `ImageFolder`.

O `ImageFolder` lê automaticamente a estrutura:

<t></t>

```
dataset_soja_split/
    train/
        classe_1/
        classe_2/
    val/
        classe_1/
        classe_2/
    test/
        classe_1/
        classe_2/

```

In [0]:
dataset_treino, dataset_val, dataset_teste = criar_datasets(
    pasta_split=DATASET_SPLIT,
    img_size=224
)


Vamos verificar:

- quantidade de imagens;
- quantidade de classes;
- nomes das classes;
- ordem das classes usada pelo PyTorch.

In [0]:
mostrar_info_datasets(
    dataset_treino,
    dataset_val,
    dataset_teste
)

In [0]:
classes = obter_classes(dataset_treino)
class_to_idx = obter_class_to_idx(dataset_treino)

print("Classes:")
print(classes)

print("\nClass to idx:")
print(class_to_idx)

Agora vamos criar os DataLoaders.

In [0]:
loader_treino, loader_val, loader_teste = criar_dataloaders(
    pasta_split=DATASET_SPLIT,
    img_size=224,
    batch_size=32,
    num_workers=0
)

Antes de seguir para o treinamento, vamos carregar um batch para confirmar se:

- as imagens estão sendo lidas corretamente;
- os tensores têm o formato esperado;
- os rótulos estão vindo corretamente.

In [0]:
imagens, labels = next(iter(loader_treino))

print("Formato das imagens:", imagens.shape)
print("Formato dos labels:", labels.shape)
print("Labels do batch:", labels)

In [0]:
print("Dataset organizado:", DATASET_ORGANIZADO)
print("Dataset com split:", DATASET_SPLIT)

# Pré-processamento concluído

Nesta etapa finalizamos:

- organização das imagens;
- padronização das classes;
- split treino/validação/teste;
- balanceamento básico;
- augmentação offline;
- criação dos datasets;
- criação e teste dos DataLoaders.